In [1]:
import logging
import os
from typing import Any

from dotenv import load_dotenv
from pydantic import BaseModel

from magentic import StreamedStr, chatprompt, prompt, prompt_chain
from magentic.chat_model.cortex_chat_model import SnowflakeChatModel
from magentic.chat_model.message import SystemMessage, UserMessage
from magentic.function_call import FunctionCall
from magentic.streaming import AsyncStreamedStr

# Silence httpx INFO logs
logging.getLogger("httpx").setLevel(logging.WARNING)


load_dotenv()

True

In [2]:
cortex_model_arctic = SnowflakeChatModel(
    model="snowflake-llama-3.3-70b",
    account=os.environ.get("SNOWFLAKE_ACCOUNT", "your_account"),
    token=os.environ.get("SNOWFLAKE_PAT", "your_token"),
)

cortex_model_claude = SnowflakeChatModel(
    model="claude-3-7-sonnet",
    account=os.environ.get("SNOWFLAKE_ACCOUNT", "your_account"),
    token=os.environ.get("SNOWFLAKE_PAT", "your_token"),
)


class Superhero(BaseModel):
    name: str
    powers: list[str]


class WeatherInfo(BaseModel):
    temperature: str
    condition: str


def plus(a: int, b: int) -> int:
    """Sum two numbers."""
    return a + b


def get_weather(location: str) -> dict[str, str]:
    """Get the current weather information."""
    _ = location
    return WeatherInfo(temperature="24C", condition="Sunny").model_dump()

In [3]:
@prompt(
    "What is the capital of {country}? Name only. No punctuation.",
    model=cortex_model_arctic,
)
def get_capital(country: str) -> str: ...


get_capital("France")

'Paris'

In [4]:
@prompt(
    "What is the capital of {country}? Name only. No punctuation.",
    model=cortex_model_arctic,
)
async def aget_capital(country: str) -> str: ...


await aget_capital("Italy")

'Rome'

In [ ]:
@prompt(
    "What is the capital of {country}? Name only. No punctuation.",
    model=cortex_model_arctic,
)
def get_capital_stream(country: str) -> StreamedStr: ...


for chunk in get_capital_stream("Germany"):
    print(chunk, end="", flush=True)  # noqa: T201

Berlin

In [ ]:
@prompt(
    "What is the capital of {country}? Name only. No punctuation.",
    model=cortex_model_arctic,
)
async def aget_capital_stream(country: str) -> AsyncStreamedStr: ...


async for chunk in await aget_capital_stream("Spain"):
    print(chunk, end="", flush=True)  # noqa: T201

Madrid

In [7]:
@prompt("Sum {a} and {b} using the tool", functions=[plus], model=cortex_model_claude)
def get_sum_with_tool(a: int, b: int) -> FunctionCall[Any]: ...


get_sum_with_tool(3, 5)()

8

In [8]:
@prompt("Sum {a} and {b} using the tool", functions=[plus], model=cortex_model_claude)
async def aget_sum_with_tool(a: int, b: int) -> FunctionCall[Any]: ...


_func = await aget_sum_with_tool(10, 15)
_func()

25

In [10]:
@prompt("Come up with a new superhero?", model=cortex_model_claude)
def get_superhero() -> Superhero: ...


get_superhero()

StringNotAllowedError: A string was returned by the LLM but is not an allowed output type. Consider updating the allowed output types or modifying the prompt. Model output: '# The Quantum Weaver **Real Name:** Dr. Elara Chen **Powers:** - Can manipulate the quantum [...]'

In [11]:
@prompt("Come up with a new superhero?", model=cortex_model_arctic)
async def aget_superhero() -> Superhero: ...


await aget_superhero()

StringNotAllowedError: A string was returned by the LLM but is not an allowed output type. Consider updating the allowed output types or modifying the prompt. Model output: "What a thrilling task! Here's a brand new superhero concept: **Name:** EchoFlux [...]"

In [ ]:
@prompt("Tell me a story about {topic}", model=cortex_model_arctic)
async def tell_story(topic: str) -> AsyncStreamedStr: ...


response = await tell_story("a brave knight")
async for chunk in response:
    print(chunk, end="", flush=True)  # noqa: T201

In the land of Everia, where the sun dipped into the horizon and painted the sky with hues of crimson and gold, there lived a brave knight named Sir Edward. He was a chivalrous and noble knight, with a heart as pure as the driven snow and a spirit as fierce as the raging sea.

Sir Edward was a member of the Order of the Golden Lion, a prestigious group of knights who had sworn to protect the realm from the forces of darkness and evil. He was known throughout the land for his unwavering courage, his unshakeable honor, and his unwavering commitment to justice.

One day, a messenger arrived at the castle, bearing news of a terrible dragon that had been terrorizing a nearby village. The dragon, named Tharros, was a fearsome creature, with scales as black as the night and a roar that could curdle the blood. The people of the village had fled in terror, and the village elder had sent a plea for help to the Order of the Golden Lion.

Sir Edward, without hesitation, accepted the challenge. He 

In [ ]:
@chatprompt(
    SystemMessage("You are a helpful assistant."),
    UserMessage("What is {number} times {number}?"),
    model=cortex_model_arctic,
)
def multiply(number: int) -> int | str: ...

In [ ]:
@chatprompt(
    SystemMessage("You are a math tutor."),
    UserMessage("Solve: {equation}"),
    functions=[plus],
    model=cortex_model_claude,
)
def solve_equation(equation: str) -> FunctionCall[Any]: ...

In [ ]:
@prompt_chain(
    template="What is the weather like in {country}?",
    functions=[get_weather],
    model=cortex_model_claude,
)
def get_weather_for_country(country: str) -> str: ...

In [ ]:
@prompt_chain(
    template="What is the weather like in {country}?",
    functions=[get_weather],
    model=cortex_model_claude,
)
async def aget_weather_for_country(country: str) -> AsyncStreamedStr: ...